In [1]:
import json
import os
from pathlib import Path

# Chemins des fichiers : fonctionne que le notebook soit exécuté depuis la racine du projet ou depuis code/
if (Path("code/data") / "train.json").exists():
    data_dir = Path("code/data")
elif (Path("data") / "train.json").exists():
    data_dir = Path("data")
else:
    data_dir = Path("code/data")  # défaut
    print(f"Attention: train.json non trouvé. Vérifiez que vous êtes dans la racine du projet ou dans code/. data_dir = {data_dir.resolve()}")
data_dir = data_dir.resolve()
train_file = data_dir / "train.json"
ct_json_dir = data_dir / "CT_json"
print(f"data_dir utilisé : {data_dir}")

data_dir utilisé : /Users/lubin/Documents/Finetuning_few_shot/code/data


In [2]:
def load_ct_data(ct_id):
    """Charge les données d'un essai clinique depuis le fichier JSON"""
    ct_file = ct_json_dir / f"{ct_id}.json"
    if not ct_file.exists():
        print(f"Warning: File {ct_file} not found")
        return None
    with open(ct_file, 'r', encoding='utf-8') as f:
        return json.load(f)

def extract_evidence(ct_data, section_id, evidence_indices):
    """Extrait les preuves d'une section spécifique selon les indices"""
    if ct_data is None:
        return ""
    
    section_data = ct_data.get(section_id, [])
    if not section_data:
        return ""
    
    # Extraire les éléments selon les indices
    evidence_items = []
    for idx in evidence_indices:
        if 0 <= idx < len(section_data):
            evidence_items.append(section_data[idx])
    
    return "\n".join(evidence_items)

def format_input_text(statement, primary_evidence, secondary_evidence=None):
    """Format: Premise first, then ask if it is in agreement with the following hypothesis, then the hypothesis."""
    if secondary_evidence:
        premise_content = f"{primary_evidence}\n\n{secondary_evidence}"
    else:
        premise_content = primary_evidence
    return (
        f"PREMISE: {premise_content}\n\n"
        f"Is this premise in agreement with the following hypothesis?\n"
        f"HYPOTHESIS: {statement}\n\n"
        f"Answer only with: Entailment or Contradiction."
    )

# Ancien prompt (même style que Gold_test_fixed_premise) : PREMISE + HYPOTHESIS sans question
SYSTEM_MSG_OLD_PROMPT = "Classify the relationship between the premise and the hypothesis. Respond with only one word: 'Entailment' or 'Contradiction'."

def format_input_text_old_prompt(statement, primary_evidence, secondary_evidence=None):
    """Format ancien (comme Gold_test_fixed_premise) : PREMISE puis HYPOTHESIS, pas de question."""
    if secondary_evidence:
        premise_content = f"{primary_evidence}\n\n{secondary_evidence}"
    else:
        premise_content = primary_evidence
    return f"PREMISE: {premise_content}\n\nHYPOTHESIS: {statement}"

In [3]:
# Les datasets seront traités dans les cellules suivantes

In [4]:
def process_dataset(input_file, output_file, format_fn=format_input_text, use_system_message=False):
    """Traite un dataset (train, dev ou test) et le formate en JSONL. format_fn: fonction de formatage; use_system_message: ajouter le system (ancien prompt)."""
    print(f"\n{'='*60}")
    print(f"Traitement de {input_file.name}...")
    print(f"{'='*60}")
    
    # Charger le fichier
    with open(input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    print(f"Nombre d'entrées dans {input_file.name}: {len(data)}")
    
    # Transformer les données au format requis
    formatted_data = []
    errors = []
    
    for entry_id, entry in data.items():
        try:
            # Récupérer les informations de base
            statement = entry.get("Statement", "")
            label = entry.get("Label", "")  # "Entailment" ou "Contradiction"
            section_id = entry.get("Section_id", "")
            primary_id = entry.get("Primary_id", "")
            entry_type = entry.get("Type", "Single")
            
            # Charger les données du CT principal
            primary_ct_data = load_ct_data(primary_id)
            if primary_ct_data is None:
                errors.append(f"Entry {entry_id}: Primary CT {primary_id} not found")
                continue
            
            # Extraire les preuves du CT principal
            primary_evidence_indices = entry.get("Primary_evidence_index", [])
            primary_evidence = extract_evidence(primary_ct_data, section_id, primary_evidence_indices)
            
            # Si c'est une comparaison, charger aussi le CT secondaire
            secondary_evidence = None
            if entry_type == "Comparison":
                secondary_id = entry.get("Secondary_id", "")
                secondary_ct_data = load_ct_data(secondary_id)
                if secondary_ct_data is None:
                    errors.append(f"Entry {entry_id}: Secondary CT {secondary_id} not found")
                    continue
                
                secondary_evidence_indices = entry.get("Secondary_evidence_index", [])
                secondary_evidence = extract_evidence(secondary_ct_data, section_id, secondary_evidence_indices)
            
            # Formater le contenu utilisateur
            user_content = format_fn(statement, primary_evidence, secondary_evidence)
            
            # Créer l'entrée au format messages (avec ou sans system)
            if use_system_message:
                formatted_entry = {
                    "messages": [
                        {"role": "system", "content": SYSTEM_MSG_OLD_PROMPT},
                        {"role": "user", "content": user_content},
                        {"role": "assistant", "content": label}
                    ]
                }
            else:
                formatted_entry = {
                    "messages": [
                        {"role": "user", "content": user_content},
                        {"role": "assistant", "content": label}
                    ]
                }
            
            formatted_data.append(formatted_entry)
            
        except Exception as e:
            errors.append(f"Entry {entry_id}: Error - {str(e)}")
            continue
    
    print(f"Nombre d'entrées formatées: {len(formatted_data)}")
    print(f"Nombre d'erreurs: {len(errors)}")
    
    if errors:
        print("\nPremières erreurs:")
        for error in errors[:10]:
            print(f"  - {error}")
    
    # Sauvegarder le dataset formaté en JSONL (une ligne JSON par entrée)
    print(f"\nSauvegarde dans {output_file}...")
    with open(output_file, 'w', encoding='utf-8') as f:
        for entry in formatted_data:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')
    
    print(f"✓ Dataset sauvegardé avec succès! ({len(formatted_data)} entrées)")
    
    return formatted_data

In [5]:
# Traiter le dataset train
train_formatted = process_dataset(train_file, data_dir / "train_formatted.jsonl")

# Afficher un exemple pour vérification
if train_formatted:
    print("\n" + "="*60)
    print("Exemple d'entrée formatée (train):")
    print("="*60)
    print(json.dumps(train_formatted[0], indent=2, ensure_ascii=False))


Traitement de train.json...
Nombre d'entrées dans train.json: 1700
Nombre d'entrées formatées: 1700
Nombre d'erreurs: 0

Sauvegarde dans /Users/lubin/Documents/Finetuning_few_shot/code/data/train_formatted.jsonl...
✓ Dataset sauvegardé avec succès! (1700 entrées)

Exemple d'entrée formatée (train):
{
  "messages": [
    {
      "role": "user",
      "content": "PREMISE: INTERVENTION 1: \n  Diagnostic (FLT PET)\n  Patients with early stage, ER positive primary breast cancer undergo FLT PET scan at baseline and 1-6 weeks after the start of standard endocrine treatment. The surgery follows 1-7 days after the second FLT PET scan.\n  Tracer used in the FLT PET (positron emission tomography) scanning procedure: [F18] fluorothymidine.\n  Positron Emission Tomography: Undergo FLT PET\n  Laboratory Biomarker Analysis: Correlative studies - Ki67 staining of the tumor tissue in the biopsy and surgical specimen.\n\nINTERVENTION 1: \n  Arm A\n  Patients receive oral capecitabine twice daily on day

In [6]:
# Traiter le dataset dev
dev_file = data_dir / "dev.json"
dev_output_file = data_dir / "dev_formatted.jsonl"
dev_formatted = process_dataset(dev_file, dev_output_file)

# Afficher un exemple pour vérification
if dev_formatted:
    print("\n" + "="*60)
    print("Exemple d'entrée formatée (dev):")
    print("="*60)
    print(json.dumps(dev_formatted[0], indent=2, ensure_ascii=False))


Traitement de dev.json...
Nombre d'entrées dans dev.json: 200
Nombre d'entrées formatées: 200
Nombre d'erreurs: 0

Sauvegarde dans /Users/lubin/Documents/Finetuning_few_shot/code/data/dev_formatted.jsonl...
✓ Dataset sauvegardé avec succès! (200 entrées)

Exemple d'entrée formatée (dev):
{
  "messages": [
    {
      "role": "user",
      "content": "PREMISE: Outcome Measurement: \n  Event-free Survival\n  Event free survival, the primary endpoint of this study, is defined as the time from randomization to the time of documented locoregional or distant recurrence, new primary breast cancer, or death from any cause.\n  Time frame: 5 years\nResults 1: \n  Arm/Group Title: Exemestane\n  Arm/Group Description: Patients receive oral exemestane (25 mg) once daily for 5 years.\n  exemestane: Given orally\n  Overall Number of Participants Analyzed: 3789\n  Measure Type: Number\n  Unit of Measure: percentage of participants  88        (87 to 89)\nResults 2: \n  Arm/Group Title: Anastrozole\n  

In [7]:
# Traiter le dataset Gold_test
gold_test_file = data_dir / "Gold_test.json"
gold_test_output_file = data_dir / "Gold_test_formatted.jsonl"
gold_test_formatted = process_dataset(gold_test_file, gold_test_output_file)

# Afficher un exemple pour vérification
if gold_test_formatted:
    print("\n" + "="*60)
    print("Exemple d'entrée formatée (Gold_test):")
    print("="*60)
    print(json.dumps(gold_test_formatted[0], indent=2, ensure_ascii=False))


Traitement de Gold_test.json...
Nombre d'entrées dans Gold_test.json: 500
Nombre d'entrées formatées: 500
Nombre d'erreurs: 0

Sauvegarde dans /Users/lubin/Documents/Finetuning_few_shot/code/data/Gold_test_formatted.jsonl...
✓ Dataset sauvegardé avec succès! (500 entrées)

Exemple d'entrée formatée (Gold_test):
{
  "messages": [
    {
      "role": "user",
      "content": "PREMISE: Exclusion Criteria:\n  History of bilateral mastectomy, osteoporosis or renal impairment.\n\nExclusion Criteria:\n  Severe claustrophobia\n\nIs this premise in agreement with the following hypothesis?\nHYPOTHESIS: Women suffering from both claustrophobia and IBS or not eligible for either the primary trial or the secondary trial.\n\nAnswer only with: Entailment or Contradiction."
    },
    {
      "role": "assistant",
      "content": "Contradiction"
    }
  ]
}


In [ ]:
# Génération des datasets avec ancien prompt (premise normale depuis les CT, pas fixée)
train_old = process_dataset(train_file, data_dir / "train_formatted_old_prompt.jsonl", format_fn=format_input_text_old_prompt, use_system_message=True)
dev_old = process_dataset(dev_file, data_dir / "dev_formatted_old_prompt.jsonl", format_fn=format_input_text_old_prompt, use_system_message=True)
gold_test_old = process_dataset(gold_test_file, data_dir / "Gold_test_formatted_old_prompt.jsonl", format_fn=format_input_text_old_prompt, use_system_message=True)
if gold_test_old:
    print("\n" + "="*60)
    print("Exemple (Gold_test ancien prompt):")
    print("="*60)
    print(json.dumps(gold_test_old[0], indent=2, ensure_ascii=False))

## Datasets avec ancien prompt (style Gold_test_fixed_premise)

Même structure de prompt que Gold_test_fixed_premise (PREMISE + HYPOTHESIS, avec message system), mais la premise vient des CT (données normales). Fichiers : train_formatted_old_prompt.jsonl, dev_formatted_old_prompt.jsonl, Gold_test_formatted_old_prompt.jsonl.

## Datasets avec ancien prompt (style Gold_test_fixed_premise)

Même structure de prompt que Gold_test_fixed_premise (PREMISE + HYPOTHESIS, avec message system), mais la premise vient des CT (données normales, pas fixée). Fichiers générés : `*_formatted_old_prompt.jsonl`.

In [ ]:
# Datasets avec ancien prompt (premise normale depuis CT, même style que Gold_test_fixed_premise)
train_old = process_dataset(train_file, data_dir / "train_formatted_old_prompt.jsonl", format_fn=format_input_text_old_prompt, use_system_message=True)
dev_old = process_dataset(dev_file, data_dir / "dev_formatted_old_prompt.jsonl", format_fn=format_input_text_old_prompt, use_system_message=True)
gold_test_old = process_dataset(gold_test_file, data_dir / "Gold_test_formatted_old_prompt.jsonl", format_fn=format_input_text_old_prompt, use_system_message=True)
if gold_test_old:
    print("\nExemple Gold_test ancien prompt:")
    print(json.dumps(gold_test_old[0], indent=2, ensure_ascii=False))